# Overcooked-AI — Pipeline Colab · Etapa 3 (PPO vs greedy)

Vertiente Colab del proyecto (comparte el 100% del código con la máquina local/GPU).

**Regla `numpy<2`:** Colab precarga numpy 2; tras instalar hay que **REINICIAR EL RUNTIME** una vez.

Orden: **Celda 1** (instalar) → **Reiniciar runtime** → **Celda 2+** (verificar, entrenar, evaluar).
El árbitro de '¿está listo?' es el **harness oficial** (Etapa 2), no la reward de entrenamiento.


## Celda 1 — Clonar repo + instalar dependencias


In [ ]:
import os
REPO_URL = 'https://github.com/Snah-s/deep_project.git'
REPO_DIR = '/content/deep_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!bash scripts/setup_colab.sh

## ⚠️ REINICIA EL RUNTIME AHORA

`Entorno de ejecución` → `Reiniciar sesión`. Luego continúa desde la **Celda 2**.
(El downgrade a `numpy<2` solo aplica tras reiniciar el intérprete.)


## Celda 2 — Verificar entorno (GATE 0)


In [ ]:
%cd /content/deep_project
!python -m pytest tests/test_env_smoke.py -q

## Celda 3 — Montar Drive (persistencia de checkpoints)

Colab pierde el disco al desconectar: los checkpoints van a Drive (obligatorio).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUT = '/content/drive/MyDrive/overcooked_ckpts'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Checkpoints ->', DRIVE_OUT)

## Celda 4 — Configuración del experimento


In [ ]:
LAYOUT = 'cramped_room'        # piloto; cambiar al layout real cuando el profe lo revele
TOTAL_TIMESTEPS = 5_000_000
N_ENVS = 8
VEC = 'subproc'                # si SubprocVecEnv falla en Colab, poner 'dummy'

## Celda 5 — Entrenamiento PPO

Guarda `best_model` por score oficial + checkpoints periódicos en Drive.
Colab free se desconecta: si pasa, re-ejecuta esta celda (el `.ipynb` sigue en el repo);
los checkpoints periódicos quedan en `DRIVE_OUT/<run>/checkpoints/`.


In [ ]:
%cd /content/deep_project
!python -m training.train_ppo \
    --config training/configs/esc1.yaml \
    --layout {LAYOUT} \
    --total-timesteps {TOTAL_TIMESTEPS} \
    --n-envs {N_ENVS} \
    --vec {VEC} \
    --device auto \
    --output-dir '{DRIVE_OUT}'

## Celda 6 — Evaluar con el harness oficial (GATE 3)

`harness(agente, greedy) ≥ harness(greedy, greedy)` y `soups_mean ≥ 1`.


In [ ]:
%cd /content/deep_project
import sys; sys.path.insert(0, 'overcooked')
from evaluation.harness import evaluate, _summary
from envs.partners import make_partner

run_name = f'esc1_greedy_{LAYOUT}'
best_path = f'{DRIVE_OUT}/{run_name}/best_model'

agent_ctor = lambda: make_partner({'type': 'checkpoint', 'path': best_path})
res  = evaluate(agent_ctor, LAYOUT, {'type': 'greedy'}, seeds=[67, 68, 69])
base = evaluate(lambda: make_partner({'type': 'greedy'}), LAYOUT, {'type': 'greedy'}, seeds=[67, 68, 69])
print('AGENTE   vs greedy:', _summary(res))
print('BASELINE greedy   :', _summary(base))
gate = (res['score_mean'] >= base['score_mean']) and (res['soups_mean'] >= 1)
print('GATE 3:', 'PASA ✅' if gate else 'NO pasa ❌')

## Celda 7 (opcional) — Curva de score del harness


In [ ]:
import json, matplotlib.pyplot as plt
hist = json.load(open(f'{DRIVE_OUT}/{run_name}/eval_history.json'))
xs = [h['timesteps'] for h in hist]; ys = [h['score_mean'] for h in hist]
plt.plot(xs, ys, marker='o'); plt.xlabel('timesteps'); plt.ylabel('harness score_mean')
plt.title(run_name); plt.grid(True); plt.show()

## Si el GATE 3 no pasa (perillas permitidas, en orden)
1. Subir `shaping.anneal_fraction` a `0.8` en `training/configs/esc1.yaml`.
2. Duplicar `TOTAL_TIMESTEPS`.

Solo esas dos, en ese orden (regla del PLAN). Registrar el cambio en `EXPERIMENTS.md`.
